In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Managed Agents API - Unified Commerce Agent Demo

### Overview

This notebook is an end-to-end demo using the **Managed Agents API** on Gemini Enterprise Agent Platform to power **Omni-AI**—a unified commerce and marketing assistant tailored for [OmniCommerce](https://omnicommerce.com/).

OmniCommerce is a leading unified commerce platform for mid-market merchants ($20M–$200M revenue). This demo shows how a single AI agent can seamlessly orchestrate operations across OmniCommerce's core product clouds:
*   **Commerce Cloud:** Checking product SKU specifications, inventory counts, and scarcity warnings.
*   **Marketing Cloud:** Leveraging customer eRFM (Recency, Frequency, Monetary Value) profiles to draft high-conversion abandoned cart emails and companion SMS messages.
*   **Service Cloud (Helpdesk):** Handling order tracking, resolving shipment delays, and structuring automated live-agent escalations.

For complete Managed Agents API documentation please visit: https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/managed-agents.

**DISCLAIMER:** This notebook uses preview APIs on Gemini Enterprise Agent Platform (`antigravity-preview-05-2026`).

### 📌 Demo Architecture Alignment for OmniCommerce

| Demo Component | Description |
|---|---|
| **Agent Persona** | **Omni-AI** (Unified Assistant bridging Commerce, Marketing & Service Clouds) |
| **Data Sources** | Local mock database `./merchant_data/` (`catalog.json`, `orders.json`, `customers.json`, `tickets.json`) |
| **Remote Skill (GCS)** | `unified_commerce_skill.md` (Mounted to `./skills` in remote execution environment) |
| **Tools Enabled** | `filesystem` (local container workspace) & `google_search` (web grounding) |
| **State Persistence** | Remote sandbox environment saved across multi-turn interactions |


## 1. Import Required Packages

Run the code cell below to import standard libraries and Google Auth packages.

In [ ]:
# @title Import Required Packages

import json
import logging
import os
import pprint
import sys
import textwrap
import time
from datetime import datetime, timezone
from typing import Any, Dict, Optional

import google.auth
import ipywidgets as widgets
import markdown
import requests
from google.auth.transport.requests import Request
from google.cloud import storage
from IPython.display import clear_output, display

# Initialize logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 2. Configuration & Core Scaffolding

This section sets up your GCP Project ID and Google Cloud Storage bucket name for mounting agent skills.

* **TokenManager**: Minting and refreshing OAuth tokens.
* **AgentFactory**: REST API wrapper for Control Plane (create, list, get, delete agents).
* **AgentConfig**: Configures Omni-AI system instructions, tools, and remote skill GCS sources.
* **AgentHarness**: Interactive multi-turn chat UI with stateful environment persistence.

In [ ]:
# @title Configure Project ID and GCS Bucket for Skill Mounting
PROJECT_ID = ""  # @param {type:"string"}
SKILL_GCS_BUCKET = ""  # @param {type:"string"}
ENDPOINT = "https://aiplatform.googleapis.com"
LOCATION = "global"
IS_COLAB = "google.colab" in sys.modules

if not PROJECT_ID:
    print("⚠️ Please set your PROJECT_ID variable.")
if not SKILL_GCS_BUCKET:
    print("⚠️ Please set your SKILL_GCS_BUCKET variable.")

In [ ]:
# @title Core Scaffolding Code for OmniCommerce Managed Agent Demo

class TokenManager:
    """Manages Google Cloud OAuth 2.0 access tokens and session states."""

    def __init__(self, is_colab: bool = False):
        self.token: Optional[str] = None
        self.expiry: Optional[datetime] = None
        self.email: Optional[str] = None
        self.is_colab: bool = is_colab

    def get_token(self) -> str:
        if self.token is None or self.is_expired():
            self.refresh_token()
        return self.token

    def is_expired(self) -> bool:
        if self.expiry is None:
            return True
        return datetime.now(timezone.utc) >= self.expiry

    def refresh_token(self) -> None:
        if self.is_colab:
            try:
                from google.colab import auth
                auth.authenticate_user()
            except ImportError:
                pass

        scopes = ["https://www.googleapis.com/auth/cloud-platform"]
        credentials, _ = google.auth.default(scopes=scopes)
        credentials.refresh(Request())
        self.token = credentials.token
        if credentials.expiry:
            self.expiry = credentials.expiry.replace(tzinfo=timezone.utc)

class AgentFactory:
    """Factory class to manage the lifecycle of Agents via REST API."""

    def __init__(self, project_id: str, endpoint: str, location: str):
        self.project_id = project_id
        self.endpoint = endpoint
        self.location = location
        self.token_manager = TokenManager()
        self.agents: Dict[str, Dict[str, Any]] = {}
        if self.project_id:
            self.list()

    def _get_headers(self) -> Dict[str, str]:
        token = self.token_manager.get_token()
        return {
            "Content-Type": "application/json; charset=utf-8",
            "Authorization": f"Bearer {token}"
        }

    def _get_base_url(self) -> str:
        return f"{self.endpoint}/v1beta1/projects/{self.project_id}/locations/{self.location}/agents"

    def _poll_operation(self, operation: Dict[str, Any]) -> None:
        op_name = operation.get("name")
        if not op_name:
            return
        operation_status_url = f"{self.endpoint}/v1beta1/{op_name}"
        while not operation.get("done"):
            time.sleep(4)
            try:
                op_response = requests.get(operation_status_url, headers=self._get_headers(), timeout=10)
                op_response.raise_for_status()
                operation = op_response.json()
            except requests.RequestException as e:
                logger.error("Polling error: %s", e)
                break

    def create(self, agent_config: Dict[str, Any]) -> Optional[requests.Response]:
        try:
            response = requests.post(self._get_base_url(), headers=self._get_headers(), json=agent_config, timeout=10)
            response.raise_for_status()
            self._poll_operation(response.json())
            self.list()
            return response
        except requests.RequestException as e:
            logger.error("Failed to create agent: %s", e)
            return None

    def list(self) -> Dict[str, Dict[str, Any]]:
        try:
            response = requests.get(self._get_base_url(), headers=self._get_headers(), params={"page_size": 10}, timeout=10)
            response.raise_for_status()
            parsed_data = response.json()
            self.agents = {agent["id"]: agent for agent in parsed_data.get("agents", [])}
        except requests.RequestException as e:
            logger.error("Failed to list agents: %s", e)
        return self.agents

    def delete(self, agent_id: str) -> Optional[requests.Response]:
        target_agent = self.agents.get(agent_id)
        if not target_agent or "name" not in target_agent:
            return None
        try:
            response = requests.delete(f"{self.endpoint}/v1beta1/{target_agent['name']}", headers=self._get_headers(), timeout=10)
            response.raise_for_status()
            self.agents.pop(agent_id, None)
            return response
        except requests.RequestException as e:
            logger.error("Failed to delete agent: %s", e)
            return None

class AgentConfig:
    """Configuration builder for the Unified Commerce Agent."""

    def __init__(self, project_id: str, skill_gcs_bucket: str):
        self.project_id = project_id
        self.skill_gcs_bucket = skill_gcs_bucket

    def get_config(self) -> Dict[str, Any]:
        system_instruction = textwrap.dedent("""\
            * You are "Omni-AI", an advanced retail assistant designed for OmniCommerce merchants to automate multi-channel engagement and service helpdesks.
            * You support three unified pillars: Commerce Cloud (inventory, specs), Marketing Cloud (segment copy, cart recovery), and Service Cloud (helpdesk tickets).
            * When starting up, parse the local repository `./merchant_data/` to load product catalogs, active customer order history, and active tickets.
            * Always prioritize the customer's eRFM profile (Recency, Frequency, Monetary Value) to recommend upsell products.
            * If a customer uses highly frustrated language or requests refunds over $150, draft a structured ticket JSON and trigger the "Escalate to Human Agent" protocol.
            
            Rule: You must always explain your reasoning (e.g., why a promotional offer matches a segment) and narrate your actions.
        """)

        config = {
            "id": "unified-commerce-agent",
            "base_agent": "antigravity-preview-05-2026",
            "description": "Unified Assistant bridging OmniCommerce Marketing, Commerce, and Service Clouds.",
            "system_instruction": system_instruction,
            "tools": [
                {"type": "filesystem"},
                {"type": "google_search"},
            ],
            "base_environment": {
                "type": "remote",
                "sources": [
                    {
                        "type": "gcs",
                        "source": f"gs://{self.skill_gcs_bucket}",
                        "target": "./skills"
                    }
                ],
                "network": {
                    "allowlist": [
                        {"domain": "*"}
                    ]
                }
            },
        }
        return config

class AgentHarness:
    def __init__(self, factory):
        self.factory = factory
        self.current_agent_config = None
        self.previous_interaction_id = None
        self.environment_id = None
        self.history_file = None
        self.history = {}
        os.makedirs("logs", exist_ok=True)

    def load_history(self):
        if self.history_file and os.path.exists(self.history_file):
            try:
                with open(self.history_file, 'r') as f:
                    return json.load(f)
            except Exception:
                return {}
        return {}

    def save_state(self, prompt):
        if self.environment_id and self.previous_interaction_id:
            now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            snippet = prompt.replace('\n', ' ')[:30] + "..."
            if self.environment_id not in self.history:
                self.history[self.environment_id] = {"last_seen": now, "interactions": {}}
            self.history[self.environment_id]["last_seen"] = now
            self.history[self.environment_id]["interactions"][self.previous_interaction_id] = {
                "snippet": snippet,
                "last_seen": now
            }
            if self.history_file:
                with open(self.history_file, 'w') as f:
                    json.dump(self.history, f, indent=2)

    def interact(self, prompt):
        api_url = f"{self.factory.endpoint}/v1beta1/projects/{self.factory.project_id}/locations/{self.factory.location}/interactions"
        headers = {
            "Authorization": f"Bearer {self.factory.token_manager.get_token()}",
            "Content-Type": "application/json"
        }
        payload = {
            "agent": self.current_agent_config.get("name"),
            "stream": True,
            "background": True,
            "store": True,
            "input": [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        }
        if self.previous_interaction_id:
            payload["previous_interaction_id"] = self.previous_interaction_id
        if self.environment_id:
            payload["environment"] = self.environment_id
        else:
            payload["environment"] = "remote"

        response = requests.post(api_url, headers=headers, json=payload, stream=True)
        self._handle_stream(response, prompt)

    def _handle_stream(self, response, prompt):
        if response.status_code != 200:
            print(f"Error HTTP {response.status_code}: {response.text}")
            return
        print("\nOmni-AI:")
        message_html = widgets.HTML(value="")
        display(message_html)
        content_blocks = {}
        text_buffers = {}

        def render_ui():
            full_text = "".join(content_blocks[idx] for idx in sorted(content_blocks.keys()))
            html_content = markdown.markdown(full_text, extensions=['nl2br', 'fenced_code', 'tables'])
            message_html.value = f"<div style='font-family: inherit; line-height: 1.5;'>{html_content}</div>"

        for line in response.iter_lines():
            if not line:
                continue
            decoded_line = line.decode('utf-8').strip()
            if decoded_line.startswith("data: "):
                data_str = decoded_line[6:]
                if data_str == "[DONE]":
                    self.save_state(prompt)
                    break
                try:
                    event_data = json.loads(data_str)
                    event_type = event_data.get("event_type")
                    index = event_data.get("index")
                    if event_type == "interaction.start":
                        interaction_id = event_data.get("interaction", {}).get("id")
                        if interaction_id:
                            self.previous_interaction_id = interaction_id
                    elif event_type == "interaction.complete":
                        env_id = event_data.get("interaction", {}).get("environment_id")
                        if env_id:
                            self.environment_id = env_id
                    if index is not None:
                        if event_type == "content.delta":
                            delta = event_data.get("delta", {})
                            if "text" in delta:
                                delta_text = delta.get("text", "")
                                text_buffers[index] = text_buffers.get(index, "") + delta_text
                                content_blocks[index] = text_buffers[index]
                                render_ui()
                except json.JSONDecodeError:
                    pass


## 3. Seed Mock Merchant Dataset

Run the code cell below to create local mock database files in `./merchant_data/` (`catalog.json`, `orders.json`, `customers.json`, `tickets.json`).

In [ ]:
# @title Seed Mock Merchant Database Files

os.makedirs("./merchant_data", exist_ok=True)

catalog = [
  {
    "sku": "UB-RUN-100",
    "name": "UltraBoost Performance Running Shoes",
    "category": "Footwear",
    "price": 140.00,
    "stock_quantity": 3,
    "colors": ["Black", "Blue", "White"],
    "sizes": [8, 9, 10, 11]
  },
  {
    "sku": "APX-JKT-200",
    "name": "Apex Thermal Winter Jacket",
    "category": "Apparel",
    "price": 220.00,
    "stock_quantity": 25,
    "colors": ["Navy", "Olive", "Charcoal"],
    "sizes": ["S", "M", "L", "XL"]
  },
  {
    "sku": "NMD-BAG-300",
    "name": "Nomad Leather Weekend Duffel Bag",
    "category": "Accessories",
    "price": 185.00,
    "stock_quantity": 2,
    "colors": ["Tan", "Espresso"]
  }
]

orders = [
  {
    "order_id": "90210",
    "customer_email": "sarah.jenkins@example.com",
    "order_date": "2026-07-02",
    "status": "Delayed in Transit",
    "carrier": "FedEx",
    "tracking_number": "TRK987654321",
    "expected_delivery": "2026-07-10",
    "items": [{"sku": "UB-RUN-100", "name": "UltraBoost Running Shoes", "size": "10", "price": 140.00}],
    "total_amount": 140.00
  }
]

customers = [
  {
    "customer_id": "CUST-101",
    "name": "Sarah Jenkins",
    "email": "sarah.jenkins@example.com",
    "erfm_segment": "VIP High-Value",
    "recency_days": 5,
    "frequency_orders": 14,
    "total_monetary_spend": 3450.00,
    "vip_tier": "Gold",
    "abandoned_cart": {
      "sku": "NMD-BAG-300",
      "name": "Nomad Leather Weekend Duffel Bag",
      "price": 185.00
    }
  }
]

with open("./merchant_data/catalog.json", "w") as f:
    json.dump(catalog, f, indent=2)
with open("./merchant_data/orders.json", "w") as f:
    json.dump(orders, f, indent=2)
with open("./merchant_data/customers.json", "w") as f:
    json.dump(customers, f, indent=2)

print("✅ Mock merchant database successfully created under ./merchant_data/")

## 4. Environment Check & Deploy Agent

Run the code below to test project permissions and deploy the `unified-commerce-agent` configuration.

In [ ]:
# @title Deploy OmniCommerce Agent Configuration

if not PROJECT_ID or not SKILL_GCS_BUCKET:
    raise ValueError("Please configure PROJECT_ID and SKILL_GCS_BUCKET above first.")

factory = AgentFactory(project_id=PROJECT_ID, endpoint=ENDPOINT, location=LOCATION)
config_builder = AgentConfig(project_id=PROJECT_ID, skill_gcs_bucket=SKILL_GCS_BUCKET)
agent_cfg = config_builder.get_config()

print("Deploying agent to Managed Agents Control Plane...")
factory.create(agent_cfg)
print("✅ Agent deployed successfully!")
print(f"Active agents: {list(factory.agents.keys())}")

## 5. Upload OmniCommerce Skill Playbook to GCS

Run the cell below to upload the Unified Commerce Agent skill playbook to your GCS bucket.

In [ ]:
# @title Upload Skill Playbook to GCS

skill_markdown = r"""
# SYSTEM DIRECTIVE: OmniCommerce_Unified_Commerce_Agent_Master

## 1. Core Identity & Persona
*   **Role:** You are "Omni-AI", an advanced retail assistant designed for OmniCommerce merchants.
*   **Tone:** Highly helpful, conversion-focused, and solutions-oriented.

## 2. Platform Architecture & Product Clouds
Categorize interactions into OmniCommerce product clouds:
*   **Commerce Cloud:** Product search, inventory count, SKU features, pricing tiers.
*   **Marketing Cloud:** Dynamic abandoned cart sequences, personalized SMS promos, eRFM customer segmentation.
*   **Service Cloud:** Order returns, status lookups, defect tracking, live-agent escalation.

## 3. Playbooks
#### A. Marketing Cloud: eRFM Cart Recovery Copy
*   **Trigger:** User asks to generate abandoned cart recovery draft.
*   **Protocol:** Extract customer eRFM profile from `./merchant_data/customers.json`, draft personalized email & companion SMS fallback.

#### B. Commerce Cloud: Inventory & Stock Checks
*   **Trigger:** Customer/merchant asks about stock or SKU availability.
*   **Protocol:** Search `./merchant_data/catalog.json`. If stock <= 5, append scarcity warning.

#### C. Service Cloud: Delayed Orders & Escalation
*   **Trigger:** Customer asks about delayed packages or refund.
*   **Protocol:** Check `./merchant_data/orders.json`. If delayed, explain & offer recovery discount code (`CARE15`). If uncooperative or refund > $150, draft JSON ticket and output `[Escalating to Service Cloud Human Queue]`.
"""

def upload_skill(bucket_name: str, content: str, blob_name: str = "unified_commerce_skill.md"):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.upload_from_string(content)
    print(f"✅ Skill successfully uploaded to gs://{bucket_name}/{blob_name}")

upload_skill(SKILL_GCS_BUCKET, skill_markdown)

## 6. Interactive Chat Harness

Run the cell below to launch the interactive Omni-AI chat session.

### Live Interactive Scenarios to Try:
1.  **Commerce Cloud (Inventory Scarcity):**
    > *"Check if the 'UltraBoost Performance Running Shoes' in size 10 are in stock. If we are running low, warn me!"*
2.  **Marketing Cloud (eRFM Personalization):**
    > *"Draft an abandoned-cart recovery email and a companion SMS for Gold VIP customer Sarah Jenkins who left a leather duffel bag in her cart."*
3.  **Service Cloud (Frictionless Escalation):**
    > *"My order #90210 has been delayed for 2 weeks. This is unacceptable, I want a refund!"*

In [ ]:
# @title Start Interactive Omni-AI Chat Session

factory = AgentFactory(project_id=PROJECT_ID, endpoint=ENDPOINT, location=LOCATION)
harness = AgentHarness(factory)
harness.current_agent_config = factory.agents.get("unified-commerce-agent")
print("Ready! Type your message to interact with Omni-AI.")